# Module 8: Poisson and Negative Binomial Regression with Harmonic Seasonality

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

[Module 2](../Module_02_Counts_Are_Not_Gaussian.md) argued that monthly incident
counts are counts and should be modelled as such. This module is the working
version of that argument: a count regression with a trend, a season and an
exposure offset, which is the right default for almost every monthly series in
a public safety dataset.

The module also makes one point that will save you from a common mistake.
**Overdispersion is usually a diagnosis about the mean, not about the
distribution.** Before you reach for a negative binomial, check whether the
Poisson is overdispersed only because something is missing from the right hand
side.

**About 35 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]          # never fit on unfinished months


CALENDAR = pd.period_range("2019-01", "2026-04", freq="M").to_timestamp()


def counts(agency_id):
    """Monthly counts on a complete calendar, so a gap stays visible as missing."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series(d["n_uof"].values, dtype=float,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


print(f"{final['agency_id'].nunique()} agencies, {final['year_month'].nunique()} months")

In [ ]:
import statsmodels.api as sm


def panel(agency_id):
    """Monthly counts with the pieces a count regression needs."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month").reset_index(drop=True)
    d = d.assign(dt=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    return d.assign(mo=d["dt"].dt.month,
                    yr=((d["dt"].dt.year - 2019) * 12 + d["dt"].dt.month - 1) / 12.0)


def harmonics(month, K):
    """K sine and cosine pairs at the annual frequency."""
    out = {}
    for k in range(1, K + 1):
        out[f"sin{k}"] = np.sin(2 * np.pi * k * month / 12)
        out[f"cos{k}"] = np.cos(2 * np.pi * k * month / 12)
    return pd.DataFrame(out)


def poisson(y, X, offset):
    return sm.GLM(y, sm.add_constant(X), family=sm.families.Poisson(),
                  offset=offset).fit()


d = panel("A001")                                  # Stonewick
y = d["n_uof"].values.astype(float)
off = np.log(d["n_arrests"].values.astype(float))  # exposure, coefficient fixed at 1
print(f"{len(y)} months, {y.mean():.1f} incidents a month on average")

## 2. Start with a trend and nothing else

The offset fixes the exposure coefficient at 1, so the model is about the rate
per arrest. Pearson dispersion should be near 1 if the Poisson fits.

In [ ]:
X = pd.DataFrame({"t": d["yr"].values})
m0 = poisson(y, X, off)
print(f"  trend {100 * (np.exp(m0.params['t']) - 1):+.1f} percent a year")
print(f"  Pearson dispersion {m0.pearson_chi2 / m0.df_resid:.2f}")

A dispersion above 2 is the textbook signal to switch to a negative binomial.
Do not switch yet. Look at where the residuals are large first.

In [ ]:
resid = pd.Series((y - m0.mu) / np.sqrt(m0.mu), index=d["mo"].values)
by_month = resid.groupby(level=0).mean()
print("average Pearson residual by calendar month:")
for mo, v in by_month.items():
    bar = "#" * int(round(abs(v) * 12))
    print(f"  {mo:2d}  {v:+.2f}  {bar}")

The residuals are not scattered. They are positive in summer and negative in
winter, which is not what a dispersion problem looks like. It is what a missing
seasonal term looks like.

## 3. Harmonics instead of monthly dummies

Two ways to add a season:

- **Eleven monthly dummies.** Maximum flexibility, eleven parameters, and a
  separate estimate for each month that borrows nothing from its neighbours.
- **Harmonics.** A sine and a cosine at the annual frequency, two parameters,
  a smooth curve, and the assumption that July resembles June and August.

For monthly public safety data the second is almost always the better trade.

In [ ]:
designs = {
    "trend only": pd.DataFrame({"t": d["yr"].values}),
    "+ 1 harmonic": pd.concat([pd.DataFrame({"t": d["yr"].values}),
                               harmonics(d["mo"].values, 1)], axis=1),
    "+ 2 harmonics": pd.concat([pd.DataFrame({"t": d["yr"].values}),
                                harmonics(d["mo"].values, 2)], axis=1),
    "+ 3 harmonics": pd.concat([pd.DataFrame({"t": d["yr"].values}),
                                harmonics(d["mo"].values, 3)], axis=1),
    "+ 11 monthly dummies": pd.concat(
        [pd.DataFrame({"t": d["yr"].values}),
         pd.get_dummies(d["mo"], prefix="m", drop_first=True).astype(float)], axis=1),
}

rows = []
for name, X in designs.items():
    m = poisson(y, X, off)
    rows.append({"design": name, "parameters": X.shape[1] + 1,
                 "AIC": round(m.aic, 1),
                 "Pearson dispersion": round(float(m.pearson_chi2 / m.df_resid), 2)})
pd.DataFrame(rows).set_index("design")

**Read the dispersion column first.** Two extra parameters took it from 2.19 to
1.09. The apparent overdispersion was the July peak, sitting in the residuals
because nothing in the model accounted for it.

Now read the AIC column. One harmonic is the best of the five. Eleven dummies
cost nine more parameters and score slightly worse.

## 4. Does the negative binomial help now?

It should not, and confirming that is worth a cell.

In [ ]:
X1 = designs["+ 1 harmonic"]
best = None
for alpha in np.arange(0.005, 0.201, 0.005):
    m = sm.GLM(y, sm.add_constant(X1), offset=off,
               family=sm.families.NegativeBinomial(alpha=alpha)).fit()
    if best is None or m.aic < best[1]:
        best = (alpha, m.aic)
print(f"  Poisson            AIC {poisson(y, X1, off).aic:7.1f}")
print(f"  negative binomial  AIC {best[1]:7.1f}   at the best alpha, {best[0]:.3f}")

The best negative binomial is at the smallest alpha on the grid and its AIC is
no better. **Once the mean is right, there is nothing left for the extra
variance parameter to do.**

This is the general rule. Fix the mean structure, then decide about the
distribution. Doing it the other way round hides a modelling error inside a
variance parameter, and the coefficients stay wrong.

## 5. When the negative binomial really is the answer

Tarnbridge has a documented month of civil unrest in June 2021.

In [ ]:
t = panel("A002")
yt = t["n_uof"].values.astype(float)
offt = np.log(t["n_arrests"].values.astype(float))
Xt = pd.concat([pd.DataFrame({"t": t["yr"].values}), harmonics(t["mo"].values, 1)], axis=1)

mt = poisson(yt, Xt, offt)
res = (yt - mt.mu) / np.sqrt(mt.mu)
print(f"  Pearson dispersion, all months     {mt.pearson_chi2 / mt.df_resid:.2f}")
print(f"  largest Pearson residual           {res.max():.1f}  at {t['year_month'][np.argmax(res)]}")

keep = (t["year_month"] != "2021-06").values
mk = poisson(yt[keep], Xt[keep], offt[keep])
print(f"  Pearson dispersion, that month out {mk.pearson_chi2 / mk.df_resid:.2f}")

A single month carries the dispersion from 1.41 to 5.92. A Pearson residual of
19 is not overdispersion: it is one extraordinary month, and it has a name and
a date.

**Three different diagnoses, three different fixes.** A residual pattern by
calendar month means a missing seasonal term. One enormous residual means an
event, which you model or exclude and document. Residuals that are simply too
big everywhere, with no pattern and no single culprit, are the case the negative
binomial exists for.

## 6. Reading the fitted season

A harmonic model's seasonal shape is not in the coefficients, it is in the
curve they trace out. Convert it before reporting it.

In [ ]:
mh = poisson(y, X1, off)
mo = np.arange(1, 13)
shape = (mh.params["sin1"] * np.sin(2 * np.pi * mo / 12)
         + mh.params["cos1"] * np.cos(2 * np.pi * mo / 12))
shape = shape - shape.mean()

amp = np.hypot(mh.params["sin1"], mh.params["cos1"])
peak = int(round((np.arctan2(mh.params["sin1"], mh.params["cos1"]) * 12 / (2 * np.pi)) % 12))
print(f"  amplitude on the log scale {amp:.3f}   (the value built in was 0.200)")
print(f"  peak month {peak}                       (the month built in was 7)")
print(f"  peak runs {100 * (np.exp(2 * amp) - 1):.0f} percent above the trough\n")
for k, v in zip(mo, shape):
    print(f"  month {k:2d}  {100 * (np.exp(v) - 1):+6.1f} percent")

Two parameters recovered an amplitude of 0.197 against a planted 0.200, and put
the peak in July. Eleven dummies recover the same shape with nine more
parameters and a rougher curve, as the third panel of the module figure shows.

## Exercise

Ashfell is the largest agency in the dataset, with about 100 incidents a month.
Run the same comparison there. Does more data change which design wins?

In [ ]:
# Fill in the blank, then run.
AGENCY = None          # try "A012"

if AGENCY:
    g = panel(AGENCY)
    yy = g["n_uof"].values.astype(float)
    oo = np.log(g["n_arrests"].values.astype(float))
    for name, K in [("trend only", 0), ("+ 1 harmonic", 1), ("+ 2 harmonics", 2),
                    ("+ 3 harmonics", 3)]:
        X = pd.DataFrame({"t": g["yr"].values})
        if K:
            X = pd.concat([X, harmonics(g["mo"].values, K)], axis=1)
        m = poisson(yy, X, oo)
        print(f"  {name:16s} AIC {m.aic:8.1f}   Pearson "
              f"{m.pearson_chi2 / m.df_resid:.2f}")
    X1g = pd.concat([pd.DataFrame({"t": g["yr"].values}),
                     harmonics(g["mo"].values, 1)], axis=1)
    bg = None
    for alpha in np.arange(0.002, 0.061, 0.002):
        z = sm.GLM(yy, sm.add_constant(X1g), offset=oo,
                   family=sm.families.NegativeBinomial(alpha=alpha)).fit()
        if bg is None or z.aic < bg[1]:
            bg = (alpha, z.aic)
    print(f"  {'NB, 1 harmonic':16s} AIC {bg[1]:8.1f}   at alpha {bg[0]:.3f}")
    rr = (yy - poisson(yy, X1g, oo).mu) / np.sqrt(poisson(yy, X1g, oo).mu)
    print(f"  largest Pearson residual {rr.max():.1f}")
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A012"
```

Two harmonics edge out one by 0.6 AIC points, which is a tie: the seasonal
shape at Ashfell is the same simple one. And again the dispersion falls
sharply when the season enters, from 4.45 to 1.73.

But at Ashfell it stops there. **The dispersion settles near 1.72 and no
number of harmonics moves it.** The largest Pearson residual is only 2.6, so
there is no single month to blame either. Both of the first two diagnoses are
ruled out, and the negative binomial improves AIC from 718.4 to 704.9 at an
alpha of 0.006.

That is the third case, and here it is real. A large agency aggregates many
neighbourhoods, shifts and unit level practices, and its month to month
variation is genuinely wider than a Poisson allows. **The negative binomial is
not papering over a modelling error, it is describing the data.** Fit it,
report the dispersion parameter, and say why you used it.

The general lesson is the order of operations, not the answer: get the mean
right first, and only then ask what the variance needs.

</details>

---

**Next:** [Module 9: Rare Events, Zero Inflation and When to Aggregate Up](Module_09_Rare_Events.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*